In [1]:
from cotton_counter.pipelines.camera_utils import CameraConfig

%reload_kedro

camera_config_raw = catalog.load("auto_camera_config")
camera_config = CameraConfig.load_partitioned(camera_config_raw)

2025-05-09 09:20:46,611 - kedro.ipython - INFO - Resolved project path as: /home/daniel/git/aerial_flower_counting.
To set a different path, run '%reload_kedro <project_root>'
2025-05-09 09:20:46,726 - py.warnings - WARNING - /home/daniel/git/aerial_flower_counting/.venv/lib/python3.10/site-packages/kedro/framework/session/session.py:267: KedroDeprecationWarning: TemplatedConfigLoader will be deprecated in Kedro 0.19. Please use the OmegaConfigLoader instead. To consult the documentation for OmegaConfigLoader, see here: https://docs.kedro.org/en/stable/configuration/advanced_configuration.html#omegaconfigloader
  warnings.warn(

2025-05-09 09:20:46,756 - py.warnings - WARNING - /home/daniel/git/aerial_flower_counting/.venv/lib/python3.10/site-packages/kedro/io/partitioned_dataset.py:200: KedroDeprecationWarning: 'PartitionedDataset' has been moved to `kedro-datasets` and will be removed in Kedro 0.19.0.
  warnings.warn(

2025-05-09 09:20:46,760 - py.warnings - WARNING - /home/daniel/gi

In [44]:
from zipfile import ZipFile

session = "2023-10-11"
artifacts = ZipFile("/home/daniel/Downloads/artifacts.zip")

In [45]:
# Group any duplicates in the list of names.
name_groups = {}
for name in artifacts.namelist():
    base_name = name
    if not base_name.startswith("DJI"):
        base_name = "_".join(base_name.split("_")[1:])

    name_groups.setdefault(base_name, []).append(name)

In [46]:
from exif import Image

def dms_to_latlon(dms):
    d, m, s = dms
    return d + m / 60 + s / 3600

def get_lat_lon(image):
    # Gets the lat/lon exif data for an image.
    exif_data = Image(image)
    latitude = dms_to_latlon(exif_data.gps_latitude)
    if exif_data.gps_latitude_ref == "S":
        latitude *= -1
    longitude = dms_to_latlon(exif_data.gps_longitude)
    if exif_data.gps_longitude_ref == "W":
        longitude *= -1

    return latitude, longitude

In [47]:
import numpy as np

session_cameras = camera_config[f"{session}_cameras"]
camera_mappings = {}

for group_names in name_groups.values():
    # Remove unaligned cameras.
    group_names = list(filter(lambda n: n.split(".")[0] in session_cameras.camera_transforms, group_names))
    
    # Read position data from the files.
    image_files = [artifacts.open(n, "r") for n in group_names]
    image_positions = [np.array(get_lat_lon(f)[::-1]) for f in image_files]
    image_positions = np.array(image_positions)

    # Compare to the metashape camera data.
    meta_positions = [session_cameras.camera_positions.get(n.split(".")[0]) for n in group_names]

    best_association = []
    for meta_pos in meta_positions:
        if meta_pos is None:
            best_association.append(None)
            continue
        meta_pos = meta_pos[:2]
        distances = np.linalg.norm(meta_pos - image_positions, axis=1)
        best_association.append(group_names[np.argmin(distances)])

    # Deal with missing positions.
    have_missing = False
    for name in group_names:
        if name not in best_association:
            assert not have_missing, "Have more than one missing position in group."
            best_association[best_association.index(None)] = name
            have_missing = True

    # Add to the mapping if needed.
    for original_name, new_name in zip(group_names, best_association):
        if original_name == new_name:
            # No change.
            continue
        camera_mappings[original_name] = new_name
        

In [48]:
print(f"Fixing {len(camera_mappings)} files.")

Fixing 392 files.


In [49]:
from tqdm import tqdm

# Swap the actual files.
with ZipFile("/home/daniel/Downloads/artifacts_fixed.zip", "w") as fixed_artifacts:
    for file_name in tqdm(artifacts.namelist()):
        source_file = camera_mappings.get(file_name, file_name)
        source_data = artifacts.open(source_file, "r").read()
        fixed_artifacts.open(file_name, "w").write(source_data)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1384/1384 [00:08<00:00, 172.56it/s]
